# Statistical Significance Tests — Report Figures

Simple tests for the 5 main report figures. All tests are non-parametric because actionability scores are heavily right-skewed (83% zero).

**Tests used:**
- Chi-square: proportion differences across groups (Figure 2)
- Mann-Whitney U: two-group comparisons (Figures 3, 4, 5)
- Kruskal-Wallis: three-or-more group comparison (Figure 5)

In [11]:
import pandas as pd
import numpy as np
from scipy.stats import kruskal, mannwhitneyu, chi2_contingency

df = pd.read_csv('output/enriched.csv')
df['source_group'] = df['source_type'].apply(
    lambda x: 'National news' if x == 'national_news' else 'Other outlets'
)
df['imperative_combined'] = (
    (df['mean_imperative_count'] > 0) | (df['mean_verbs_imperative_count'] > 0)
).astype(int)

def sig(p):
    return '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'ns'))

results = []

def mwu(label, figure, a, b, group1, group2):
    u, p = mannwhitneyu(a, b, alternative='two-sided')
    results.append({
        'Figure': figure,
        'Comparison': label,
        'Group 1': f'{group1} (n={len(a)}, mean={a.mean():.2f}%)',
        'Group 2': f'{group2} (n={len(b)}, mean={b.mean():.2f}%)',
        'Test': 'Mann-Whitney U',
        'Statistic': f'U={u:.0f}',
        'p': round(p, 4),
        'Sig': sig(p)
    })

def kw(label, figure, groups, group_names):
    h, p = kruskal(*groups)
    means = ', '.join(f'{n}={g.mean():.2f}%' for n, g in zip(group_names, groups))
    results.append({
        'Figure': figure,
        'Comparison': label,
        'Group 1': means,
        'Group 2': '',
        'Test': 'Kruskal-Wallis',
        'Statistic': f'H={h:.3f}, df={len(groups)-1}',
        'p': round(p, 4),
        'Sig': sig(p)
    })

def chi2(label, figure, col, group_col, groups):
    sub = df[df[group_col].isin(groups)].copy()
    sub['present'] = (sub[col] > 0).astype(int)
    ct = pd.crosstab(sub[group_col], sub['present'])
    for c in [0, 1]:
        if c not in ct.columns: ct[c] = 0
    chi2_val, p, _, _ = chi2_contingency(ct[[0, 1]])
    pcts = ', '.join(
        f'{g}={(sub.loc[sub[group_col]==g,"present"].mean()*100):.0f}%' for g in groups
    )
    results.append({
        'Figure': figure,
        'Comparison': label,
        'Group 1': pcts,
        'Group 2': '',
        'Test': 'Chi-square',
        'Statistic': f'χ²={chi2_val:.2f}, df={len(groups)-1}',
        'p': round(p, 4),
        'Sig': sig(p)
    })

print('Setup complete — running tests')

Setup complete — running tests


In [12]:
# ── Figure 1: Descriptive only — no test needed ───────────────────────────────
s = df['actionability_percentage']
desc = pd.DataFrame([
    ('N articles', len(s)),
    ('N zero (0%)', f"{(s==0).sum()} ({(s==0).mean()*100:.1f}%)"),
    ('N non-zero', f"{(s>0).sum()} ({(s>0).mean()*100:.1f}%)"),
    ('Corpus mean', f"{s.mean():.2f}%"),
    ('Overall median', f"{s.median():.1f}%"),
    ('Non-zero median', f"{s[s>0].median():.1f}%"),
    ('Max', f"{s.max():.1f}%"),
], columns=['Statistic', 'Value'])

print('FIGURE 1 — Range of Actionability (descriptive)')
print(desc.to_string(index=False))

FIGURE 1 — Range of Actionability (descriptive)
      Statistic       Value
     N articles         607
    N zero (0%) 506 (83.4%)
     N non-zero 101 (16.6%)
    Corpus mean       1.99%
 Overall median        0.0%
Non-zero median       10.0%
            Max       40.0%


In [ ]:
# ── Figure 2: Chi-square for each PADM component across languages ─────────────
components = {
    'Imperative signals': 'imperative_combined',
    'Short-term urgency': 'mean_short_term_count',
    'Spatial anchors':    'mean_spatial_count',
    'Advice-framing':     'mean_advice',
}
for label, col in components.items():
    chi2(f'{label} — EN vs ES vs PT', '2', col, 'language', ['en', 'es', 'pt'])




In [16]:
# ── Figure 3: National news vs other outlets ──────────────────────────────────
nat = df.loc[df['source_group'] == 'National news', 'actionability_percentage'].dropna()
oth = df.loc[df['source_group'] == 'Other outlets', 'actionability_percentage'].dropna()
mwu('National news vs Other outlets', '3', nat, oth, 'National news', 'Other outlets')

In [17]:
# ── Figure 4: Frame × source type — accountability is the only frame where
#              source type makes a difference
for frame in ['impact', 'response', 'accountability', 'recovery']:
    sub = df[df['dominant_frame'] == frame]
    a = sub.loc[sub['source_group'] == 'National news', 'actionability_percentage'].dropna()
    b = sub.loc[sub['source_group'] == 'Other outlets', 'actionability_percentage'].dropna()
    if len(a) < 3 or len(b) < 3:
        results.append({
            'Figure': '4', 'Comparison': f'{frame} — National vs Other',
            'Group 1': f'National (n={len(a)})', 'Group 2': f'Other (n={len(b)})',
            'Test': 'Mann-Whitney U', 'Statistic': 'n/a (n<3)', 'p': '-', 'Sig': '-'
        })
        continue
    mwu(f'{frame} — National vs Other', '4', a, b, f'{frame}/National', f'{frame}/Other')

In [18]:
# ── Figure 5: Cluster differences ────────────────────────────────────────────
# Kruskal-Wallis across all 3 clusters
cluster_ids = sorted(df['data_cluster_id'].dropna().unique())
groups = [df.loc[df['data_cluster_id'] == c, 'actionability_percentage'].dropna() for c in cluster_ids]
names = [f'Cluster {int(c)}' for c in cluster_ids]
kw('Across all 3 clusters', '5', groups, names)

# Cluster 1 (actionable advisory) vs all others combined
# identify which cluster is the actionable one (highest mean)
means = {c: df.loc[df['data_cluster_id']==c,'actionability_percentage'].mean() for c in cluster_ids}
top_cluster = max(means, key=means.get)
c1 = df.loc[df['data_cluster_id'] == top_cluster, 'actionability_percentage'].dropna()
rest = df.loc[df['data_cluster_id'] != top_cluster, 'actionability_percentage'].dropna()
mwu(f'Cluster {int(top_cluster)} (Actionable Advisory) vs all others', '5', c1, rest,
    f'Cluster {int(top_cluster)}', 'All other clusters')

In [19]:
# ── Final table ───────────────────────────────────────────────────────────────
table = pd.DataFrame(results)
print('=' * 80)
print('STATISTICAL SIGNIFICANCE — ALL TESTS')
print('=' * 80)
print(table.to_string(index=False))
print()
print('Sig: *** p<0.001  ** p<0.01  * p<0.05  ns = not significant')

STATISTICAL SIGNIFICANCE — ALL TESTS
Figure                                    Comparison                                            Group 1                                  Group 2           Test       Statistic      p Sig
     2           Imperative signals — EN vs ES vs PT                             en=34%, es=10%, pt=16%                                              Chi-square  χ²=27.67, df=2 0.0000 ***
     2           Short-term urgency — EN vs ES vs PT                             en=51%, es=56%, pt=56%                                              Chi-square   χ²=0.65, df=2 0.7234  ns
     2              Spatial anchors — EN vs ES vs PT                             en=68%, es=88%, pt=92%                                              Chi-square  χ²=36.19, df=2 0.0000 ***
     2               Advice-framing — EN vs ES vs PT                               en=7%, es=13%, pt=5%                                              Chi-square   χ²=9.34, df=2 0.0094  **
     3                Nation